## Machine Learning Group Project -  Transit Reliability and Neighbourhood Equity in Toronto

Hi Team, since I proposed this TTC delay in Toronto Neighbourhood project, I wanted to make sure I can access all the data, and that the data links up so we can build the dataset we need.

I am developing on Google Colab, so I have to make sure certain modules are installed on the machine I'm running on.

In [ ]:
# Install the Toronto Open Data wrapper
!pip install toronto-open-data

# Install Geopandas (crucial for the spatial join)
!pip install geopandas

This next is specific to running on Google Colab.

In [ ]:
# in google colab, mount the google drive to access the data
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# TTC Bus Delay Data



I downloaded the TTC Bus Delay Data off of the following website:

https://open.toronto.ca/dataset/ttc-bus-delay-data/

I downloaded the
"TTC Bus Delay Data since 2025" in JSON Format

In [ ]:
import pandas as pd
import json

# This is my subway delay data
pathToFile = r"/content/drive/My Drive/Colab Notebooks/GroupProject/TorontoDataFiles/"

fileName = 'TTC Bus Delay Data since 2025.json'

delay_dataset = pd.read_json(pathToFile+fileName)

delay_dataset.head(10)

,_id,Date,Line,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Vehicle
0,1,2025-01-01,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442
1,2,2025-01-01,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,None,0
2,3,2025-01-01,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,None,8546
3,4,2025-01-01,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693
4,5,2025-01-01,34 EGLINTON EAST,03:05,Wednesday,EGLINTON AND DON MILLS,MFUI,20,40,W,8801
5,6,2025-01-01,320 YONGE,03:15,Wednesday,YONGE AND ELM,MFUS,0,0,None,0
6,7,2025-01-01,307 BATHURST,03:20,Wednesday,BATHURST AND DUNDAS,MFUS,0,0,N,0
7,8,2025-01-01,320 YONGE,03:23,Wednesday,YONGE AND SHUTER,MFSAN,5,10,N,3468
8,9,2025-01-01,112 WEST MALL,03:39,Wednesday,RENFORTH STATION,SFO,0,0,None,3306
9,10,2025-01-01,39 FINCH EAST,04:38,Wednesday,FINCH AND MCCOWAN,EFO,16,32,W,3144


In [ ]:
delay_dataset.shape


(69037, 11)

So I picked the 4th row, that has a delay at OVERLEA AND THORNCLIFF	for 100 Flemingdon Park. I want to see if I can get to the Neighborhood it is from.

# TTC Routes and Schedules Data



I downloaded the data at:
https://open.toronto.ca/dataset/ttc-routes-and-schedules/


In [ ]:
import pandas as pd
import requests
import io
from zipfile import ZipFile

# Example 1: Loading a local ZIP containing a CSV
# df = pd.read_csv('your_file.zip')

# Example 2: Loading from a URL (like the TTC Routes and Schedules data metadata found earlier)
zip_url = 'https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/7795b45e-e65a-4465-81fc-c36b9dfff169/resource/cfb6b2b8-6191-41e3-bda1-b175c51148cb/download/opendata_ttc_schedules.zip'

# Note: This specific TTC zip contains multiple text files (GTFS format).
# To load a specific file from inside the zip:
r = requests.get(zip_url)
with ZipFile(io.BytesIO(r.content)) as z:
    # List files in the zip
    print("Files in ZIP:", z.namelist())
    # Load 'routes.txt' which is a CSV-like format
    with z.open('routes.txt') as f:
        df_routes = pd.read_csv(f)
    with z.open('stops.txt') as f:
        df_route_stops = pd.read_csv(f)
    with z.open('stop_times.txt') as f:
        df_stop_times = pd.read_csv(f)
    with z.open('trips.txt') as f:
        df_trips = pd.read_csv(f)

display(df_routes.head())

Files in ZIP: ['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'routes.txt', 'shapes.txt', 'stops.txt', 'stop_times.txt', 'trips.txt']


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,1,1,1,Line 1 (Yonge-University),NaN,1,NaN,D5C82B,000000
1,10,1,10,Van Horne,NaN,3,NaN,ED1C24,FFFFFF
2,100,1,100,Flemingdon Park,NaN,3,NaN,ED1C24,FFFFFF
3,101,1,101,Downsview Park,NaN,3,NaN,ED1C24,FFFFFF
4,102,1,102,Markham Rd,NaN,3,NaN,ED1C24,FFFFFF


So now I have the TTC routes and schedules, i can see if I can map the delay at OVERLEA and THORNCLIFFE to any route stops.

In [ ]:
#df_route_stops.head()

In [ ]:
# find an entry in df_route_stops with stop_name column
# having 'Overlea' and 'Thorncliff'
df_route_stops[df_route_stops['stop_name'].str.contains('Overlea') &
               df_route_stops['stop_name'].str.contains('Thorncliff')]

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
215,8011,8011,Overlea Blvd at Thorncliffe Park Dr (East),NaN,43.707250,-79.344028,NaN,NaN,NaN,NaN,NaN,1
486,8012,8012,Overlea Blvd at Thorncliffe Park Dr (West),NaN,43.705197,-79.349856,NaN,NaN,NaN,NaN,NaN,1
3663,4520,4520,Overlea Blvd at Thorncliffe Park Dr (West),NaN,43.704776,-79.349853,NaN,NaN,NaN,NaN,NaN,1
4108,8361,8361,Thorncliffe Park Dr East at Overlea Blvd South...,NaN,43.707006,-79.343507,NaN,NaN,NaN,NaN,NaN,1
6296,4518,4518,Overlea Blvd at Thorncliffe Park Dr (East) Wes...,NaN,43.707457,-79.344198,NaN,NaN,NaN,NaN,NaN,1
6990,8352,8352,Thorncliffe Park Dr (East) at Overlea Blvd,NaN,43.706926,-79.343188,NaN,NaN,NaN,NaN,NaN,1


So as you can see, we found multiple bus stops at Overlead and Thorncliffe. So we can probably do more data cleaning to get the exact stop it was at, or we can take the mean longitude and latitude and use that to map it to a neighborhood.

# Toronto Neighborhoods

https://open.toronto.ca/dataset/neighbourhoods/

I downloaded this 158-neighbourhoods proile

https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/fc443770-ef0a-4025-9c2c-2cb558bfab00/resource/0719053b-28b7-48ea-b863-068823a93aaa/download/Neighbourhoods%20-%204326.geojson

I just used the longitude and latitude of stop_id 8011 (the first entry above) to see if it works.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# 1. Load the Toronto 158-Neighborhood boundaries
gdf_neighborhoods = gpd.read_file(pathToFile + "Neighbourhoods - 4326.geojson")

# 2. Define your point (Longitude first for Shapely!)
lon, lat = -79.344028, 43.707250
my_point = Point(lon, lat)

# 3. Create a GeoDataFrame for your point(s)
gdf_point = gpd.GeoDataFrame([{'geometry': my_point}], crs="EPSG:4326")

# 4. Perform the Spatial Join
result = gpd.sjoin(gdf_point, gdf_neighborhoods, how="left", predicate="within")

# 5. Extract the Neighborhood Name and Number
neighborhood_name = result['AREA_NAME'].iloc[0]
# Usually 'AREA_SHORT_CODE' contains the ID/Number (e.g., 43)
neighborhood_number = result['AREA_SHORT_CODE'].iloc[0]

print(f"The coordinate maps to: {neighborhood_name} (Neighborhood #{neighborhood_number})")

# Display first few columns of the result to see all available attributes
display(result.head())

The coordinate maps to: Thorncliffe Park (Neighborhood #055)


,geometry,index_right,_id,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID
0,POINT (-79.34403 43.70725),129,130,2502237,26022752,None,055,055,Thorncliffe Park,Thorncliffe Park (55),Neighbourhood Improvement Area,NIA,17826801.0


# Toronto Neighborhood Profile Data

Now I will load the Neighbourhood profile data, from the census 2021.

You must get the 158 neighbourhood model after 2021, else it is not current and won't match our transit data from 2025.

https://open.toronto.ca/dataset/neighbourhood-profiles/

In [ ]:
df_hoods = pd.read_excel(pathToFile+"neighbourhood-profiles-2021-158-model.xlsx", skiprows= 0, header = 0, index_col = 0, sheet_name="hd2021_census_profile")

df_hoods.head(20)

,West Humber-Clairville,Mount Olive-Silverstone-Jamestown,Thistletown-Beaumond Heights,Rexdale-Kipling,Elms-Old Rexdale,Kingsview Village-The Westway,Willowridge-Martingrove-Richview,Humber Heights-Westmount,Edenbridge-Humber Valley,Princess-Rosethorn,...,Harbourfront-CityPlace,St Lawrence-East Bayfront-The Islands,Church-Wellesley,Downtown Yonge East,Bay-Cloverhill,Yonge-Bay Corridor,Junction-Wallace Emerson,Dovercourt Village,North Toronto,South Eglinton-Davisville
Neighbourhood Name,,,,,,,,,,,,,,,,,,,,,
Neighbourhood Number,1,2,3,4,5,6,7,8,9,10,...,165,166,167,168,169,170,171,172,173,174
TSNS 2020 Designation,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area,Not an NIA or Emerging Neighbourhood,Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,...,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood
Total - Age groups of the population - 25% sample data,33300,31345,9850,10375,9355,22005,22445,10005,15190,11170,...,28135,31285,22320,17700,16670,12645,23180,12380,15885,22735
0 to 14 years,4295,5690,1495,1575,1610,3915,3500,1370,2070,1750,...,2065,2285,895,1055,745,970,3075,1365,1315,2190
0 to 4 years,1460,1650,505,505,440,1245,1065,395,520,545,...,1030,1045,495,480,370,500,1135,445,535,910
5 to 9 years,1345,1860,540,615,480,1325,1190,430,740,590,...,655,690,230,325,255,270,1020,430,390,670
10 to 14 years,1485,2175,455,455,685,1350,1240,540,815,610,...,385,550,175,245,120,200,925,490,390,610
15 to 64 years,23640,21490,6615,6950,6355,14385,13865,6245,9650,7110,...,24540,25015,19315,14695,13815,10820,17200,9040,12780,17495
15 to 19 years,1860,2280,570,515,635,1245,1175,525,885,755,...,365,600,465,390,535,340,750,460,465,575


In [ ]:
# display hardcoded for now
df_hoods.iloc[:, 49]


,Thorncliffe Park
Neighbourhood Name,
Neighbourhood Number,55
TSNS 2020 Designation,Neighbourhood Improvement Area
Total - Age groups of the population - 25% sample data,20400
0 to 14 years,4975
0 to 4 years,1585
...,...
"Total - Eligibility and instruction in the minority official language, for the population in private households born between 2003 and 2015 (inclusive) - 25% sample data",4120
Children eligible for instruction in the minority official language,190
Eligible children who have been instructed in the minority official language at the primary or secondary level in Canada,120


Probably need to transpose this Neighborhood data, but if you scroll to see additional columns, or look at it in Google Sheets or Excel, you will see that Thorncliffe Park is neighbourhood #55.  Yay!

So now we can perform a spatial join with the TTC bus delay, and build our data.


# RISKS

The TTC Bus delay uses intersection string in all caps.  It would have been better if it used stop_id.  So we would need to go through all the rows, and map each one to a neighborhood.  I would suggest that we start mapping what we can, and that can be our dataset.  We have 69k rows of data, maybe it is ok we lose a few?  

